In [ ]:
import networkx as nx
import random
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

# ---------------------------
#  Load graph
# ---------------------------
G = nx.read_graphml("hetionet.graphml")
G_u = G

#XGBoost

#####the following code predicts the type of link exists between nodes

In [ ]:
import networkx as nx
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
from collections import Counter



# ---------------------------
#  Helper: extract node type
# ---------------------------
def get_node_type(node_id):
    return node_id.split("::")[0]

# ---------------------------
#  Edge features (toy example: node degrees)
# ---------------------------
def edge_to_features(u, v, G):
    return [G.degree(u), G.degree(v)]

# ---------------------------
# Build dataset (all edges, multi-class)
# ---------------------------
edges, labels = [], []
for u, v, d in G.edges(data=True):
    edges.append((u, v))
    labels.append(d.get("relation", "unknown"))

# Encode edge types
le = LabelEncoder()
y = le.fit_transform(labels)
num_classes = len(le.classes_)

# Filter out rare classes with only 1 example
counts = Counter(y)
valid_classes = [c for c, n in counts.items() if n >= 2]
mask = np.isin(y, valid_classes)

X = np.array([edge_to_features(u, v, G) for u, v in edges])[mask]
y = y[mask]
num_classes = len(np.unique(y))

# ---------------------------
# Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------
# Train multi-class XGBoost
# ---------------------------
clf = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    num_class=num_classes,
    use_label_encoder=False
)
clf.fit(X_train, y_train)

# ---------------------------
#  Evaluate
# ---------------------------
y_prob = clf.predict_proba(X_test)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

print("\n Multi-class edge-type prediction (XGBoost):")
print(f"Accuracy  = {acc:.4f}")
print(f"Precision = {precision:.3f}")
print(f"Recall    = {recall:.3f}")
print(f"F1 Score  = {f1:.3f}")


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [00:29:18] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

🌍 Multi-class edge-type prediction (XGBoost):
Accuracy  = 0.6233
Precision = 0.367
Recall    = 0.242
F1 Score  = 0.260


In [4]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=valid_classes, zero_division=0
)

for i, cls in enumerate(le.classes_):
    if i not in valid_classes:
        continue
    idx = valid_classes.index(i)
    print(f"Class '{cls}': P={precision[idx]:.3f}, R={recall[idx]:.3f}, F1={f1[idx]:.3f}, Support={support[idx]}")


Class 'AdG': P=0.435, R=0.059, F1=0.103, Support=20448
Class 'AeG': P=0.721, R=0.968, F1=0.826, Support=105282
Class 'AuG': P=0.437, R=0.005, F1=0.011, Support=19570
Class 'CbG': P=0.473, R=0.375, F1=0.418, Support=2314
Class 'CdG': P=0.402, R=0.118, F1=0.182, Support=4220
Class 'CpD': P=0.105, R=0.026, F1=0.041, Support=78
Class 'CrC': P=0.441, R=0.388, F1=0.413, Support=1297
Class 'CtD': P=0.321, R=0.119, F1=0.174, Support=151
Class 'CuG': P=0.368, R=0.163, F1=0.226, Support=3751
Class 'DaG': P=0.205, R=0.022, F1=0.039, Support=2525
Class 'DdG': P=0.225, R=0.076, F1=0.114, Support=1525
Class 'DlA': P=0.382, R=0.121, F1=0.184, Support=720
Class 'DpS': P=0.417, R=0.206, F1=0.275, Support=671
Class 'DrD': P=0.000, R=0.000, F1=0.000, Support=109
Class 'DuG': P=0.218, R=0.083, F1=0.120, Support=1546
Class 'GcG': P=0.360, R=0.240, F1=0.288, Support=12338
Class 'GiG': P=0.409, R=0.346, F1=0.375, Support=29433
Class 'GpPW': P=0.437, R=0.420, F1=0.429, Support=16874
Class 'Gr>G': P=0.612, R=0

##### the following code predicts if a link is there or not using XGBoost

In [ ]:
# ---------------------------
#  Load graph
# ---------------------------
G = nx.read_graphml("hetionet.graphml")
G_u = G

# ---------------------------
# Helper: extract node type
# ---------------------------
def get_node_type(node_id):
    return node_id.split("::")[0]

# ---------------------------
# Edge features (toy: degrees)
# ---------------------------
def edge_to_features(u, v, G):
    return [G.degree(u), G.degree(v)]

# ---------------------------
# Build balanced dataset for one relation
# ---------------------------
def build_dataset(G, relation, seed=42):
    random.seed(seed)
    all_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get("relation") == relation]
    if len(all_edges) == 0:
        return None, None, None

    random.shuffle(all_edges)
    n = len(all_edges)
    n_train = int(0.8 * n)
    n_val   = int(0.1 * n)

    train_pos, val_pos, test_pos = all_edges[:n_train], all_edges[n_train:n_train+n_val], all_edges[n_train+n_val:]

    nodes = list(G.nodes())
    def sample_negatives(num):
        negs = []
        forbidden = set(G.edges())
        while len(negs) < num:
            u, v = random.sample(nodes, 2)
            if (u, v) not in forbidden and (v, u) not in forbidden:
                negs.append((u, v))
        return negs

    train_neg = sample_negatives(len(train_pos))
    val_neg   = sample_negatives(len(val_pos))
    test_neg  = sample_negatives(len(test_pos))

    def to_df(edges, label):
        feats = [edge_to_features(u, v, G) for u, v in edges]
        return pd.DataFrame(feats, columns=["deg_u", "deg_v"]).assign(label=label, u=[u for u,v in edges], v=[v for u,v in edges])

    df_train = pd.concat([to_df(train_pos, 1), to_df(train_neg, 0)]).sample(frac=1).reset_index(drop=True)
    df_val   = pd.concat([to_df(val_pos, 1), to_df(val_neg, 0)]).sample(frac=1).reset_index(drop=True)
    df_test  = pd.concat([to_df(test_pos, 1), to_df(test_neg, 0)]).sample(frac=1).reset_index(drop=True)

    return df_train, df_val, df_test

# ---------------------------
#  Train + evaluate per relation
# ---------------------------
results = []
overall_y_true, overall_y_prob = [], []

edge_types = set(nx.get_edge_attributes(G, "relation").values())
print("[INFO] Found relation types:", len(edge_types))

for rel in edge_types:
    tr, va, te = build_dataset(G, rel)
    if tr is None or tr.empty:
        continue

    X_train, y_train = tr[["deg_u", "deg_v"]].values, tr["label"].values
    X_val, y_val     = va[["deg_u", "deg_v"]].values, va["label"].values
    X_test, y_test   = te[["deg_u", "deg_v"]].values, te["label"].values

    clf = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        use_label_encoder=False
    )
    clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    y_prob = clf.predict_proba(X_test)[:, 1]
    overall_y_true.extend(y_test)
    overall_y_prob.extend(y_prob)

    # Sweep thresholds for best F1
    best_f1, best_thr, best_p, best_r = 0, 0.5, 0, 0
    for thr in np.linspace(0.05, 0.95, 19):
        y_pred = (y_prob >= thr).astype(int)
        p = precision_score(y_test, y_pred, zero_division=0)
        r = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr, best_p, best_r = f1, thr, p, r

    auc = roc_auc_score(y_test, y_prob)

    src_type = get_node_type(te["u"].iloc[0])
    tgt_type = get_node_type(te["v"].iloc[0])
    results.append((src_type, rel, tgt_type, auc, best_p, best_r, best_f1, best_thr))

# ---------------------------
# 3) Print results
# ---------------------------
print("\n📊 Final evaluation per edge type (balanced, type-consistent negatives):")
for src, rel, tgt, auc, p, r, f1, thr in results:
    print(f"('{src}', '{rel}', '{tgt}'): AUC={auc:.4f}, P={p:.3f}, R={r:.3f}, F1={f1:.3f}, Thr={thr:.2f}")

# ---------------------------
# 4) Overall metrics
# ---------------------------
overall_auc = roc_auc_score(overall_y_true, overall_y_prob)
best_f1, best_thr, best_p, best_r = 0, 0.5, 0, 0
for thr in np.linspace(0.05, 0.95, 19):
    y_pred = (np.array(overall_y_prob) >= thr).astype(int)
    p = precision_score(overall_y_true, y_pred, zero_division=0)
    r = recall_score(overall_y_true, y_pred, zero_division=0)
    f1 = f1_score(overall_y_true, y_pred, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr, best_p, best_r = f1, thr, p, r

print("\n🧮 Overall performance across all relations (balanced negatives):")
print(f"AUC = {overall_auc:.4f}")
print(f"Precision = {best_p:.3f}")
print(f"Recall = {best_r:.3f}")
print(f"F1 Score = {best_f1:.3f}")


[INFO] Found relation types: 20


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [20:32:53] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [20:32:59] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [20:33:07] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [20:33:48] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [20:33:57] WARNING: /w


📊 Final evaluation per edge type (balanced, type-consistent negatives):
('Pathway', 'DrD', 'Anatomy'): AUC=0.9494, P=0.871, R=0.991, F1=0.927, Thr=0.05
('Biological Process', 'DlA', 'Molecular Function'): AUC=0.9858, P=0.918, R=0.985, F1=0.950, Thr=0.30
('Gene', 'GcG', 'Gene'): AUC=0.9513, P=0.840, R=0.976, F1=0.903, Thr=0.45
('Gene', 'Gr>G', 'Side Effect'): AUC=0.9951, P=0.963, R=0.984, F1=0.973, Thr=0.50
('Compound', 'CdG', 'Gene'): AUC=0.9947, P=0.956, R=0.988, F1=0.972, Thr=0.40
('Anatomy', 'AuG', 'Gene'): AUC=0.9999, P=1.000, R=1.000, F1=1.000, Thr=0.05
('Disease', 'DpS', 'Symptom'): AUC=0.9900, P=0.957, R=0.990, F1=0.973, Thr=0.50
('Compound', 'GiG', 'Biological Process'): AUC=0.9811, P=0.913, R=0.958, F1=0.935, Thr=0.50
('Anatomy', 'DaG', 'Gene'): AUC=0.9890, P=0.946, R=0.955, F1=0.951, Thr=0.60
('Disease', 'DuG', 'Gene'): AUC=0.9994, P=0.993, R=0.995, F1=0.994, Thr=0.35
('Side Effect', 'CbG', 'Side Effect'): AUC=0.9659, P=0.883, R=0.964, F1=0.922, Thr=0.50
('Biological Process

In [ ]:
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.2.0+cpu.html


#FFNN

##### the following code predicts the type of link exists between nodes using FFNN

In [ ]:
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_recall_fscore_support
from collections import Counter

# ---------------------------
# 1) Load graph
# ---------------------------
G = nx.read_graphml("hetionet.graphml")

# ---------------------------
# 2) Helper: extract node type
# ---------------------------
def get_node_type(node_id):
    return node_id.split("::")[0]

# ---------------------------
# 3) Edge features (toy example: degrees)
# ---------------------------
def edge_to_features(u, v, G):
    return [G.degree(u), G.degree(v)]

# ---------------------------
# 4) Build full dataset (all edges with relation labels)
# ---------------------------
edges = [(u, v, d.get("relation")) for u, v, d in G.edges(data=True)]
edges = [(u, v, rel) for u, v, rel in edges if rel is not None]

X = np.array([edge_to_features(u, v, G) for u, v, _ in edges])
y = [rel for _, _, rel in edges]

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Filter classes with at least 2 examples
counts = Counter(y_enc)
valid_classes = [cls for cls, cnt in counts.items() if cnt >= 2]
mask = np.isin(y_enc, valid_classes)
X = X[mask]
y_enc = y_enc[mask]

num_classes = len(np.unique(y_enc))

# Train/Val/Test split (80/10/10)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y_enc, test_size=0.1, random_state=42, stratify=y_enc
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.125, random_state=42, stratify=y_trainval
)

print(f"Dataset sizes → Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# ---------------------------
# 5) FFNN Model
# ---------------------------
class FFNN(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, output_dim=num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x  # CrossEntropyLoss expects logits

# ---------------------------
# 6) DataLoaders
# ---------------------------
BATCH_SIZE = 1024
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def make_loader(X, y):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

train_loader = make_loader(X_train, y_train)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

# ---------------------------
# 7) Training
# ---------------------------
model = FFNN(input_dim=X.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

EPOCHS = 20
best_val_acc = 0
best_model_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    avg_loss = total_loss / len(X_train)

    # Validation
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            pred = out.argmax(dim=1)
            val_preds.extend(pred.cpu().numpy())
            val_true.extend(yb.cpu().numpy())
    val_acc = accuracy_score(val_true, val_preds)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict()
    print(f"Epoch {epoch:02d} | Train Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

# Load best model
model.load_state_dict(best_model_state)

# ---------------------------
# 8) Evaluation on Test Set
# ---------------------------
model.eval()
test_preds, test_true = [], []
test_probs = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        prob = F.softmax(out, dim=1)
        pred = out.argmax(dim=1)
        test_preds.extend(pred.cpu().numpy())
        test_true.extend(yb.cpu().numpy())
        test_probs.extend(prob.cpu().numpy())

# Overall metrics
print("\n📊 Test set performance:")
print(classification_report(test_true, test_preds, target_names=[le.classes_[i] for i in range(num_classes)]))
overall_acc = accuracy_score(test_true, test_preds)
overall_f1 = f1_score(test_true, test_preds, average="macro")
print(f"Overall Accuracy = {overall_acc:.4f}, Macro F1 = {overall_f1:.4f}")



Dataset sizes → Train: 965951, Val: 137994, Test: 275987
Epoch 01 | Train Loss: 88.2619 | Val Acc: 0.2799
Epoch 02 | Train Loss: 9.3128 | Val Acc: 0.5713
Epoch 03 | Train Loss: 2.1286 | Val Acc: 0.5616
Epoch 04 | Train Loss: 1.5324 | Val Acc: 0.5866
Epoch 05 | Train Loss: 1.3583 | Val Acc: 0.5588
Epoch 06 | Train Loss: 1.2894 | Val Acc: 0.5967
Epoch 07 | Train Loss: 1.2020 | Val Acc: 0.5969
Epoch 08 | Train Loss: 1.1822 | Val Acc: 0.5988
Epoch 09 | Train Loss: 1.1651 | Val Acc: 0.5982
Epoch 10 | Train Loss: 1.1524 | Val Acc: 0.5808
Epoch 11 | Train Loss: 2.0744 | Val Acc: 0.3858
Epoch 12 | Train Loss: 1.8806 | Val Acc: 0.5203
Epoch 13 | Train Loss: 1.2753 | Val Acc: 0.5969
Epoch 14 | Train Loss: 1.2047 | Val Acc: 0.5864
Epoch 15 | Train Loss: 1.2013 | Val Acc: 0.5896
Epoch 16 | Train Loss: 1.2070 | Val Acc: 0.5921
Epoch 17 | Train Loss: 1.2005 | Val Acc: 0.5909
Epoch 18 | Train Loss: 1.1986 | Val Acc: 0.5740
Epoch 19 | Train Loss: 1.1927 | Val Acc: 0.5905
Epoch 20 | Train Loss: 1.1976 

#####the following code predicts if there is a link exists between nodes

In [ ]:
# -----------------------------
# 1) & 2) Reproducibility, device, load graph (same as before)
# -----------------------------
import torch, torch.nn as nn, torch.nn.functional as F, networkx as nx, random, numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

seed = 42
random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

graph_path = "hetionet.graphml"
G = nx.read_graphml(graph_path)
print(f"Loaded graph with {len(G.nodes)} nodes and {len(G.edges)} edges.")

# -----------------------------
# 3) Node / edge mappings
# -----------------------------
node_type_map, node_id_map, node_idx_by_type = defaultdict(list), {}, defaultdict(dict)
for nid, data in G.nodes(data=True):
    if 'kind' not in data: continue
    ntype = data['kind']
    idx = len(node_type_map[ntype])
    node_type_map[ntype].append(nid)
    node_id_map[nid] = (ntype, idx)
    node_idx_by_type[ntype][nid] = idx

edge_type_map = defaultdict(list)
for src, dst, attr in G.edges(data=True):
    rel = attr.get('metaedge', attr.get('relation', None))
    if not rel or (src not in node_id_map) or (dst not in node_id_map): continue
    src_t, src_i = node_id_map[src]; dst_t, dst_i = node_id_map[dst]
    edge_type_map[(src_t, rel, dst_t)].append((src_i, dst_i))

print(f"✅ Node types: {list(node_type_map.keys())}")
print(f"✅ Total edge types: {len(edge_type_map)}")

# -----------------------------
# 4) Build homogeneous graph tensors
# -----------------------------
type_offsets, global_id_map, offset = {}, {}, 0
for ntype, nodes in node_type_map.items():
    type_offsets[ntype] = offset
    for local_idx in range(len(nodes)):
        global_id_map[(ntype, local_idx)] = offset + local_idx
    offset += len(nodes)
num_nodes = offset

edge_type_keys = list(edge_type_map.keys())
rel2id = {etype: i for i, etype in enumerate(edge_type_keys)}
num_relations = len(edge_type_keys)

all_src, all_dst, all_rel = [], [], []
for etype, edges in edge_type_map.items():
    rel_id = rel2id[etype]; src_type, _, dst_type = etype
    for s_local, d_local in edges:
        all_src.append(type_offsets[src_type] + s_local)
        all_dst.append(type_offsets[dst_type] + d_local)
        all_rel.append(rel_id)

edge_index = torch.tensor([all_src, all_dst], dtype=torch.long)
edge_type  = torch.tensor(all_rel, dtype=torch.long)
print(f"Homogeneous graph -> nodes: {num_nodes}, edges: {edge_index.size(1)}")

# -----------------------------
# 5) Per-relation splits
# -----------------------------
edge_type_splits = {}; keep_count=0
for target_edge_type in edge_type_keys:
    rel_id = rel2id[target_edge_type]; mask=(edge_type==rel_id)
    edges = edge_index[:, mask]; n_edges = edges.size(1)
    if n_edges<100: continue
    perm=torch.randperm(n_edges)
    n_train=int(0.8*n_edges); n_val=int(0.1*n_edges)
    train_idx, val_idx, test_idx = perm[:n_train], perm[n_train:n_train+n_val], perm[n_train+n_val:]
    splits = {"train": edges[:,train_idx].to(device), "val": edges[:,val_idx].to(device),
              "test": edges[:,test_idx].to(device), "all": edges.to(device)}
    edge_type_splits[target_edge_type]=splits; keep_count+=1
print(f"Prepared splits for {keep_count} relation types.")

# -----------------------------
# 6) FNN model
# -----------------------------
class FNN(nn.Module):
    def __init__(self, num_nodes, emb_dim=128, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, emb_dim)
        self.fc1 = nn.Linear(2*emb_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
    def forward(self, src_idx, dst_idx):
        src_emb = self.emb(src_idx)
        dst_emb = self.emb(dst_idx)
        x = torch.cat([src_emb, dst_emb], dim=-1)
        x = F.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x)).squeeze(-1)

# -----------------------------
# 7) Negative sampling
# -----------------------------
def build_pools_and_forbidden(edge_type_splits, node_type_map, type_offsets):
    rel_resources={}
    for rel_key, splits in edge_type_splits.items():
        src_t, _, dst_t = rel_key
        src_pool=list(range(type_offsets[src_t], type_offsets[src_t]+len(node_type_map[src_t])))
        dst_pool=list(range(type_offsets[dst_t], type_offsets[dst_t]+len(node_type_map[dst_t])))
        forbidden=set((int(s), int(d)) for s,d in zip(splits["all"][0].tolist(), splits["all"][1].tolist()))
        rel_resources[rel_key]={"src_pool": src_pool,"dst_pool": dst_pool,"forbidden": forbidden}
    return rel_resources

def sample_type_consistent_negs(num_samples, src_pool, dst_pool, forbidden_set, device, max_tries=10):
    if num_samples==0: return torch.empty(2,0,dtype=torch.long,device=device)
    src_pool_t = torch.tensor(src_pool, device=device); dst_pool_t = torch.tensor(dst_pool, device=device)
    neg_src_list, neg_dst_list=[],[]; remaining=num_samples; tries=0
    while remaining>0 and tries<max_tries:
        k = max(remaining*2,1024)
        s_idx=torch.randint(0,len(src_pool_t),(k,),device=device)
        d_idx=torch.randint(0,len(dst_pool_t),(k,),device=device)
        s,d = src_pool_t[s_idx], dst_pool_t[d_idx]
        pairs = torch.stack([s,d],dim=1).detach().cpu().tolist()
        mask=torch.tensor([tuple(p) not in forbidden_set for p in pairs], device=device)
        s,d=s[mask], d[mask]; take=min(remaining,s.numel())
        if take>0: neg_src_list.append(s[:take]); neg_dst_list.append(d[:take]); remaining-=take
        tries+=1
    if remaining>0:
        pad_s=src_pool_t[torch.randint(0,len(src_pool_t),(remaining,),device=device)]
        pad_d=dst_pool_t[torch.randint(0,len(dst_pool_t),(remaining,),device=device)]
        neg_src_list.append(pad_s); neg_dst_list.append(pad_d)
    neg_src, neg_dst = torch.cat(neg_src_list), torch.cat(neg_dst_list)
    return torch.stack([neg_src, neg_dst], dim=0)

# -----------------------------
# 8) Edge scoring
# -----------------------------
def edge_scores_fnn(model, edges):
    return model(edges[0], edges[1])

# -----------------------------
# 9) Threshold / evaluation (fixed device handling)
# -----------------------------
@torch.no_grad()
def best_threshold_on_val_fnn(model, rel_key, splits, rel_resources, grid=None):
    device = model.emb.weight.device
    grid = grid or np.linspace(0.05,0.95,19)
    pos_e = splits["val"]; N = pos_e.size(1)
    if N==0: return 0.5
    src_pool = rel_resources[rel_key]["src_pool"]
    dst_pool = rel_resources[rel_key]["dst_pool"]
    forbidden = rel_resources[rel_key]["forbidden"]
    neg_e = sample_type_consistent_negs(N, src_pool, dst_pool, forbidden, device)
    y_true = torch.cat([torch.ones(N,device=device), torch.zeros(N,device=device)])
    scores = torch.cat([edge_scores_fnn(model,pos_e), edge_scores_fnn(model,neg_e)]).cpu().numpy()
    best_t, best_f1 = 0.5, -1
    for t in grid:
        f1 = f1_score(y_true.cpu().numpy(), (scores>t).astype(int))
        if f1>best_f1: best_f1,best_t = f1,t
    return float(best_t)

@torch.no_grad()
def evaluate_balanced_per_relation_fnn(model, rel_key, splits, rel_resources, threshold=0.5):
    device = model.emb.weight.device
    pos_e = splits["test"]; N = pos_e.size(1)
    if N==0: return float('nan'), float('nan'), float('nan'), float('nan')
    src_pool = rel_resources[rel_key]["src_pool"]
    dst_pool = rel_resources[rel_key]["dst_pool"]
    forbidden = rel_resources[rel_key]["forbidden"]
    neg_e = sample_type_consistent_negs(N, src_pool, dst_pool, forbidden, device)
    y_true = torch.cat([torch.ones(N,device=device), torch.zeros(N,device=device)])
    scores = torch.cat([edge_scores_fnn(model,pos_e), edge_scores_fnn(model,neg_e)]).cpu().numpy()
    auc = roc_auc_score(y_true.cpu().numpy(), scores)
    preds = (scores>threshold).astype(int)
    p = precision_score(y_true.cpu().numpy(), preds)
    r = recall_score(y_true.cpu().numpy(), preds)
    f1 = f1_score(y_true.cpu().numpy(), preds)
    return auc,p,r,f1

# -----------------------------
# 10) Training
# -----------------------------
model = FNN(num_nodes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
EPOCHS=30
rel_resources = build_pools_and_forbidden(edge_type_splits, node_type_map, type_offsets)

for epoch in range(1,EPOCHS+1):
    model.train(); optimizer.zero_grad(); loss_total=0
    for rel_key,splits in edge_type_splits.items():
        pos_e=splits["train"]; N=pos_e.size(1)
        if N==0: continue
        src_pool,dst_pool,forbidden = rel_resources[rel_key]["src_pool"],rel_resources[rel_key]["dst_pool"],rel_resources[rel_key]["forbidden"]
        neg_e = sample_type_consistent_negs(N, src_pool, dst_pool, forbidden, device)
        pos_scores = edge_scores_fnn(model,pos_e); neg_scores=edge_scores_fnn(model,neg_e)
        pos_loss = -(pos_scores+1e-15).log().mean(); neg_loss=-(1.0-neg_scores+1e-15).log().mean()
        (pos_loss+neg_loss).backward(); loss_total += (pos_loss+neg_loss).item()
    optimizer.step()
    if epoch%5==0 or epoch==1: print(f"Epoch {epoch:02d} | Loss: {loss_total:.4f}")

# -----------------------------
# 11) Per-relation evaluation
# -----------------------------
per_rel_threshold={}
for rel_key,splits in edge_type_splits.items():
    t=best_threshold_on_val_fnn(model,rel_key,splits,rel_resources)
    per_rel_threshold[rel_key]=t

print("\n📊 Final evaluation per edge type:")
for rel_key,splits in edge_type_splits.items():
    t=per_rel_threshold.get(rel_key,0.5)
    auc,p,r,f1=evaluate_balanced_per_relation_fnn(model,rel_key,splits,rel_resources,threshold=t)
    print(f"{rel_key}: AUC={auc:.4f}, P={p:.3f}, R={r:.3f}, F1={f1:.3f}, Thr={t:.2f}")


Using device: cuda
Loaded graph with 47033 nodes and 1379933 edges.
✅ Node types: ['Anatomy', 'Biological Process', 'Cellular Component', 'Compound', 'Disease', 'Gene', 'Molecular Function', 'Pathway', 'Pharmacologic Class', 'Side Effect', 'Symptom']
✅ Total edge types: 19
Homogeneous graph -> nodes: 47031, edges: 1379932
Prepared splits for 19 relation types.
Epoch 01 | Loss: 26.4300
Epoch 05 | Loss: 23.5070
Epoch 10 | Loss: 20.4265
Epoch 15 | Loss: 17.7188
Epoch 20 | Loss: 15.8809
Epoch 25 | Loss: 14.4939
Epoch 30 | Loss: 13.5942

📊 Final evaluation per edge type:
('Anatomy', 'AdG', 'Gene'): AUC=0.9688, P=0.904, R=0.987, F1=0.944, Thr=0.65
('Anatomy', 'AeG', 'Gene'): AUC=0.9619, P=0.893, R=0.961, F1=0.925, Thr=0.25
('Anatomy', 'AuG', 'Gene'): AUC=0.9690, P=0.905, R=0.988, F1=0.945, Thr=0.65
('Compound', 'CrC', 'Compound'): AUC=0.7836, P=0.657, R=0.845, F1=0.739, Thr=0.20
('Compound', 'CtD', 'Disease'): AUC=0.8099, P=0.667, R=0.895, F1=0.764, Thr=0.15
('Compound', 'CbG', 'Gene'): AUC=

In [ ]:
# -----------------------------
# 12) Overall evaluation across all relations
# -----------------------------
all_pos, all_neg = [], []
for rel_key,splits in edge_type_splits.items():
    pos_e = splits["test"]; N = pos_e.size(1)
    if N==0: continue
    src_pool,dst_pool,forbidden = rel_resources[rel_key]["src_pool"],rel_resources[rel_key]["dst_pool"],rel_resources[rel_key]["forbidden"]
    neg_e = sample_type_consistent_negs(N, src_pool, dst_pool, forbidden, device)
    all_pos.append(pos_e); all_neg.append(neg_e)

if all_pos:
    all_pos = torch.cat(all_pos, dim=1)
    all_neg = torch.cat(all_neg, dim=1)

    with torch.no_grad():
        scores_pos = edge_scores_fnn(model, all_pos)
        scores_neg = edge_scores_fnn(model, all_neg)

    y_true = torch.cat([torch.ones(scores_pos.numel(), device=device),
                        torch.zeros(scores_neg.numel(), device=device)]).cpu().numpy()
    scores_all = torch.cat([scores_pos, scores_neg]).cpu().numpy()

    auc_overall = roc_auc_score(y_true, scores_all)
    thr_overall = 0.5  # simple fixed threshold; can grid-search if needed
    preds = (scores_all > thr_overall).astype(int)
    p_overall = precision_score(y_true, preds)
    r_overall = recall_score(y_true, preds)
    f1_overall = f1_score(y_true, preds)

    print("\n🧮 Overall performance across all relations (balanced negatives):")
    print(f"AUC = {auc_overall:.4f}")
    print(f"Precision = {p_overall:.3f}")
    print(f"Recall = {r_overall:.3f}")
    print(f"F1 Score = {f1_overall:.3f}")
else:
    print("No test edges available for overall evaluation.")



🧮 Overall performance across all relations (balanced negatives):
AUC = 0.9472
Precision = 0.876
Recall = 0.897
F1 Score = 0.886


#GAT

## Link prediction binary

In [ ]:
# ===============================
# 0) Install required packages
# ===============================
# !pip install torch torch_geometric scikit-learn networkx

# ===============================
# 1) Device & reproducibility
# ===============================
import torch, random, numpy as np
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===============================
# 2) Load graph
# ===============================
import networkx as nx
G = nx.read_graphml("hetionet.graphml")  # replace with your graph file


In [ ]:
# ===============================
# 0) Imports & device
# ===============================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from collections import defaultdict
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===============================
# 1) Graph preprocessing (G assumed to be loaded)
# ===============================
# Node mappings
node_type_map = defaultdict(list)
node_id_map = {}
node_idx_by_type = defaultdict(dict)
for nid, data in G.nodes(data=True):
    ntype = data.get("kind", "Unknown")
    idx = len(node_type_map[ntype])
    node_type_map[ntype].append(nid)
    node_id_map[nid] = (ntype, idx)
    node_idx_by_type[ntype][nid] = idx

# Edge mappings
edge_type_map = defaultdict(list)
for u, v, data in G.edges(data=True):
    rel = data.get("metaedge", data.get("relation", None))
    if not rel or u not in node_id_map or v not in node_id_map:
        continue
    u_t, u_i = node_id_map[u]
    v_t, v_i = node_id_map[v]
    edge_type_map[(u_t, rel, v_t)].append((u_i, v_i))

print(f"Node types: {list(node_type_map.keys())}")
print(f"Edge types: {len(edge_type_map)}")

# ===============================
# 2) Build homogeneous graph tensors
# ===============================
type_offsets = {}
global_id_map = {}
offset = 0
for ntype, nodes in node_type_map.items():
    type_offsets[ntype] = offset
    for i in range(len(nodes)):
        global_id_map[(ntype, i)] = offset + i
    offset += len(nodes)
num_nodes = offset

edge_type_keys = list(edge_type_map.keys())
rel2id = {etype: i for i, etype in enumerate(edge_type_keys)}
num_relations = len(edge_type_keys)

all_src, all_dst, all_rel = [], [], []
for etype, edges in edge_type_map.items():
    rel_id = rel2id[etype]
    src_type, _, dst_type = etype
    for s_local, d_local in edges:
        all_src.append(type_offsets[src_type] + s_local)
        all_dst.append(type_offsets[dst_type] + d_local)
        all_rel.append(rel_id)

edge_index = torch.tensor([all_src, all_dst], dtype=torch.long, device=device)
edge_type  = torch.tensor(all_rel, dtype=torch.long, device=device)
print(f"Graph tensor -> nodes: {num_nodes}, edges: {edge_index.size(1)}")

# ===============================
# 3) Per-relation train/val/test splits
# ===============================
edge_type_splits = {}
for etype in edge_type_keys:
    rel_id = rel2id[etype]
    mask = (edge_type == rel_id)
    edges = edge_index[:, mask]
    n_edges = edges.size(1)
    if n_edges < 50:  # skip too small
        continue
    perm = torch.randperm(n_edges)
    n_train = int(0.8 * n_edges)
    n_val = int(0.1 * n_edges)
    train_idx = perm[:n_train]
    val_idx = perm[n_train:n_train+n_val]
    test_idx = perm[n_train+n_val:]
    edge_type_splits[etype] = {
        "train": edges[:, train_idx],
        "val": edges[:, val_idx],
        "test": edges[:, test_idx],
        "all": edges
    }

# ===============================
# 4) GAT model
# ===============================
class GATNet(nn.Module):
    def __init__(self, num_nodes, out_dim=64, heads=2):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, 64)
        self.gat1 = GATConv(64, 64, heads=heads, concat=True)
        self.gat2 = GATConv(64*heads, out_dim, heads=1, concat=True)

    def forward(self, node_ids, edge_index):
        x = self.emb(node_ids)
        x = F.relu(self.gat1(x, edge_index))
        x = self.gat2(x, edge_index)
        return x

def edge_scores(z, eidx):
    return torch.sigmoid((z[eidx[0]] * z[eidx[1]]).sum(dim=-1))

# ===============================
# 5) Negative sampling
# ===============================
def build_pools_and_forbidden(edge_type_splits, node_type_map, type_offsets):
    rel_resources = {}
    for rel_key, splits in edge_type_splits.items():
        src_t, _, dst_t = rel_key
        src_pool = list(range(type_offsets[src_t], type_offsets[src_t] + len(node_type_map[src_t])))
        dst_pool = list(range(type_offsets[dst_t], type_offsets[dst_t] + len(node_type_map[dst_t])))
        forbidden = set((int(s), int(d)) for s, d in zip(splits["all"][0].tolist(), splits["all"][1].tolist()))
        rel_resources[rel_key] = {"src_pool": src_pool, "dst_pool": dst_pool, "forbidden": forbidden}
    return rel_resources

def sample_type_consistent_negs(num_samples, src_pool, dst_pool, forbidden_set, device):
    src_pool_t = torch.tensor(src_pool, device=device)
    dst_pool_t = torch.tensor(dst_pool, device=device)
    neg_src, neg_dst = [], []
    while len(neg_src) < num_samples:
        s = src_pool_t[torch.randint(0, len(src_pool_t), (num_samples*2,))]
        d = dst_pool_t[torch.randint(0, len(dst_pool_t), (num_samples*2,))]
        pairs = [(int(si), int(di)) for si, di in zip(s, d)]
        filtered = [p for p in pairs if p not in forbidden_set]
        if len(filtered) > 0:
            ns = [p[0] for p in filtered][:num_samples - len(neg_src)]
            nd = [p[1] for p in filtered][:num_samples - len(neg_dst)]
            neg_src.extend(ns)
            neg_dst.extend(nd)
    return torch.tensor([neg_src, neg_dst], device=device)

rel_resources = build_pools_and_forbidden(edge_type_splits, node_type_map, type_offsets)

# ===============================
# 6) Train unified GAT
# ===============================
node_ids = torch.arange(num_nodes, device=device)
model = GATNet(num_nodes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
EPOCHS = 20

for epoch in range(1, EPOCHS+1):
    model.train()
    optimizer.zero_grad()
    z = model(node_ids, edge_index)
    pos_edges_all, neg_edges_all = [], []
    for rel_key, splits in edge_type_splits.items():
        pos_e = splits["train"]
        if pos_e.size(1) == 0:
            continue
        N = pos_e.size(1)
        neg_e = sample_type_consistent_negs(N,
                    rel_resources[rel_key]["src_pool"],
                    rel_resources[rel_key]["dst_pool"],
                    rel_resources[rel_key]["forbidden"],
                    device=device)
        pos_edges_all.append(pos_e)
        neg_edges_all.append(neg_e)
    pos_edges_all = torch.cat(pos_edges_all, dim=1)
    neg_edges_all = torch.cat(neg_edges_all, dim=1)
    pos_scores = edge_scores(z, pos_edges_all)
    neg_scores = edge_scores(z, neg_edges_all)
    loss = -(pos_scores+1e-15).log().mean() - (1-neg_scores+1e-15).log().mean()
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")


In [ ]:

# ===============================
# 7) Evaluation per relation
# ===============================
model.eval()
results = []
with torch.no_grad():
    z = model(node_ids, edge_index)
    for rel_key, splits in edge_type_splits.items():
        N = splits["test"].size(1)
        if N == 0: continue
        neg_e = sample_type_consistent_negs(N,
                    rel_resources[rel_key]["src_pool"],
                    rel_resources[rel_key]["dst_pool"],
                    rel_resources[rel_key]["forbidden"],
                    device=device)
        y_true = torch.cat([torch.ones(N, device=device), torch.zeros(N, device=device)])
        scores = torch.cat([edge_scores(z, splits["test"]), edge_scores(z, neg_e)])
        y = y_true.cpu().numpy()
        s = scores.cpu().numpy()
        auc = roc_auc_score(y, s)
        thr = 0.5
        preds = (s > thr).astype(int)
        p = precision_score(y, preds, zero_division=0)
        r = recall_score(y, preds, zero_division=0)
        f1 = f1_score(y, preds, zero_division=0)
        results.append((rel_key, auc, p, r, f1, thr))

print("\n📊 Final evaluation per edge type:")
for rel_key, auc, p, r, f1, thr in results:
    print(f"{rel_key}: AUC={auc:.4f}, P={p:.3f}, R={r:.3f}, F1={f1:.3f}, Thr={thr:.2f}")

# ===============================
# 8) Overall evaluation
# ===============================
all_pos, all_neg = [], []
with torch.no_grad():
    for rel_key, splits in edge_type_splits.items():
        N = splits["test"].size(1)
        if N == 0: continue
        neg_e = sample_type_consistent_negs(N,
                    rel_resources[rel_key]["src_pool"],
                    rel_resources[rel_key]["dst_pool"],
                    rel_resources[rel_key]["forbidden"],
                    device=device)
        all_pos.append(splits["test"])
        all_neg.append(neg_e)

all_pos = torch.cat(all_pos, dim=1)
all_neg = torch.cat(all_neg, dim=1)

with torch.no_grad():
    z = model(node_ids, edge_index)
    scores_pos = edge_scores(z, all_pos)
    scores_neg = edge_scores(z, all_neg)

y_true = torch.cat([torch.ones(scores_pos.numel(), device=device),
                    torch.zeros(scores_neg.numel(), device=device)])
scores_all = torch.cat([scores_pos, scores_neg], dim=0)
thr_overall = 0.5
preds_overall = (scores_all > thr_overall).int()

auc_overall = roc_auc_score(y_true.cpu().numpy(), scores_all.cpu().numpy())
p_overall   = precision_score(y_true.cpu().numpy(), preds_overall.cpu().numpy(), zero_division=0)
r_overall   = recall_score(y_true.cpu().numpy(), preds_overall.cpu().numpy(), zero_division=0)
f1_overall  = f1_score(y_true.cpu().numpy(), preds_overall.cpu().numpy(), zero_division=0)

print("\n🧮 Overall performance across all relations:")
print(f"AUC = {auc_overall:.4f}, Precision = {p_overall:.3f}, Recall = {r_overall:.3f}, F1 = {f1_overall:.3f}")



📊 Final evaluation per edge type:
('Anatomy', 'AdG', 'Gene'): AUC=0.7816, P=0.619, R=0.954, F1=0.751, Thr=0.50
('Anatomy', 'AeG', 'Gene'): AUC=0.7900, P=0.618, R=0.937, F1=0.745, Thr=0.50
('Anatomy', 'AuG', 'Gene'): AUC=0.8048, P=0.616, R=0.961, F1=0.751, Thr=0.50
('Compound', 'CrC', 'Compound'): AUC=0.7828, P=0.609, R=0.948, F1=0.741, Thr=0.50
('Compound', 'CtD', 'Disease'): AUC=0.6494, P=0.566, R=0.842, F1=0.677, Thr=0.50
('Compound', 'CbG', 'Gene'): AUC=0.6259, P=0.606, R=0.637, F1=0.621, Thr=0.50
('Compound', 'CuG', 'Gene'): AUC=0.5033, P=0.519, R=0.413, F1=0.460, Thr=0.50
('Compound', 'CdG', 'Gene'): AUC=0.4863, P=0.507, R=0.411, F1=0.454, Thr=0.50
('Compound', 'CpD', 'Disease'): AUC=0.6246, P=0.545, R=0.769, F1=0.638, Thr=0.50
('Disease', 'DdG', 'Gene'): AUC=0.5902, P=0.564, R=0.768, F1=0.650, Thr=0.50
('Disease', 'DpS', 'Symptom'): AUC=0.5967, P=0.533, R=0.881, F1=0.664, Thr=0.50
('Disease', 'DlA', 'Anatomy'): AUC=0.6574, P=0.553, R=0.886, F1=0.681, Thr=0.50
('Disease', 'DrD', 

##predicting the type of link using GAT

In [ ]:
# -----------------------------
# 1️⃣ Imports
# -----------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import networkx as nx
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from torch_geometric.nn import GATConv

# -----------------------------
# 2️⃣ Load Hetionet Graph
# -----------------------------
G_nx = nx.read_graphml("hetionet.graphml")
print(f"✅ Hetionet: {G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges")

# -----------------------------
# 3️⃣ Map nodes and encode edge types
# -----------------------------
node_id_map = {nid:i for i, nid in enumerate(G_nx.nodes())}
num_nodes = len(node_id_map)

edges_src, edges_dst, edge_types = [], [], []
for u, v, d in G_nx.edges(data=True):
    edges_src.append(node_id_map[u])
    edges_dst.append(node_id_map[v])
    edge_types.append(d['relation'])

le = LabelEncoder()
rel_ids = le.fit_transform(edge_types)
num_classes = len(le.classes_)

edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
edge_type = torch.tensor(rel_ids, dtype=torch.long)

# -----------------------------
# 4️⃣ Node features (degree-based)
# -----------------------------
x = torch.tensor([[G_nx.degree(n)] for n in G_nx.nodes()], dtype=torch.float)

# -----------------------------
# 5️⃣ Train/Val/Test edge split
# -----------------------------
num_edges = edge_index.size(1)
idx = np.arange(num_edges)
np.random.shuffle(idx)

n_train = int(0.8 * num_edges)
n_val = int(0.1 * num_edges)

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train+n_val]
test_idx = idx[n_train+n_val:]

train_edges = edge_index[:, train_idx]
train_labels = edge_type[train_idx]
val_edges = edge_index[:, val_idx]
val_labels = edge_type[val_idx]
test_edges = edge_index[:, test_idx]
test_labels = edge_type[test_idx]

# -----------------------------
# 6️⃣ Define Edge GAT Model
# -----------------------------
class GAT_EdgePredictor(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_heads, num_classes, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList()
        self.num_heads = num_heads

        # input layer
        self.layers.append(GATConv(in_dim, hidden_dim, heads=num_heads))
        # hidden layers
        for _ in range(num_layers-1):
            self.layers.append(GATConv(hidden_dim*num_heads, hidden_dim, heads=num_heads))

        # final classifier for edges
        self.classifier = nn.Linear(hidden_dim*num_heads*2, num_classes)

    def forward(self, x, edge_index, edge_pairs):
        h = x
        for layer in self.layers:
            h = F.elu(layer(h, edge_index).flatten(1))  # flatten heads

        src, dst = edge_pairs[0], edge_pairs[1]
        edge_emb = torch.cat([h[src], h[dst]], dim=1)
        return self.classifier(edge_emb)

# -----------------------------
# 7️⃣ Training Setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = x.to(device)
edge_index = edge_index.to(device)
train_edges = train_edges.to(device)
train_labels = train_labels.to(device)
val_edges = val_edges.to(device)
val_labels = val_labels.to(device)
test_edges = test_edges.to(device)
test_labels = test_labels.to(device)

model = GAT_EdgePredictor(in_dim=1, hidden_dim=64, num_heads=4, num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

# DataLoader for edges
BATCH_SIZE = 10000
train_dataset = TensorDataset(train_edges.T, train_labels)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# -----------------------------
# 8️⃣ Training Loop
# -----------------------------
EPOCHS = 10
best_val_f1 = 0
best_state = None

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    for batch_edges, batch_labels in train_loader:
        optimizer.zero_grad()
        batch_edges = batch_edges.T.to(device)  # shape [2, batch_size]
        batch_labels = batch_labels.to(device)

        outputs = model(x, edge_index, batch_edges)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    with torch.no_grad():
        val_out = model(x, edge_index, val_edges)
        val_pred = val_out.argmax(dim=1)
        val_f1 = f1_score(val_labels.cpu(), val_pred.cpu(), average='macro')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict()

    print(f"Epoch {epoch:02d} | Loss: {total_loss:.4f} | Val Macro F1: {val_f1:.4f}")

# Load best model
model.load_state_dict(best_state)

# -----------------------------
# 9️⃣ Edge-wise Evaluation
# -----------------------------
@torch.no_grad()
def edge_classification_report(model, x, edge_index, edge_pairs, edge_labels, le):
    model.eval()
    out = model(x, edge_index, edge_pairs)
    preds = out.argmax(dim=1).cpu().numpy()
    labels = edge_labels.cpu().numpy()

    print("\n📊 Edge-wise Classification Report:")
    print(classification_report(labels, preds, target_names=le.classes_, digits=3))
    acc = accuracy_score(labels, preds)
    print(f"Overall Accuracy: {acc:.4f}")



✅ Hetionet: 47033 nodes, 1379933 edges
Epoch 01 | Loss: 1282.6663 | Val Macro F1: 0.1644
Epoch 02 | Loss: 110.1544 | Val Macro F1: 0.3361
Epoch 03 | Loss: 93.0317 | Val Macro F1: 0.3672
Epoch 04 | Loss: 91.3580 | Val Macro F1: 0.3441
Epoch 05 | Loss: 89.7560 | Val Macro F1: 0.3784
Epoch 06 | Loss: 90.0167 | Val Macro F1: 0.3659
Epoch 07 | Loss: 85.4397 | Val Macro F1: 0.4275
Epoch 08 | Loss: 82.2561 | Val Macro F1: 0.4235
Epoch 09 | Loss: 85.4131 | Val Macro F1: 0.3971
Epoch 10 | Loss: 80.9572 | Val Macro F1: 0.4448




In [16]:
@torch.no_grad()
def edge_classification_report(model, x, edge_index, edge_pairs, edge_labels, le):
    model.eval()
    out = model(x, edge_index, edge_pairs)
    preds = out.argmax(dim=1).cpu().numpy()
    labels = edge_labels.cpu().numpy()

    # All class indices
    all_labels = np.arange(len(le.classes_))

    print("\n📊 Edge-wise Classification Report:")
    print(classification_report(
        labels, preds,
        labels=all_labels,
        target_names=le.classes_,
        digits=3,
        zero_division=0
    ))

    acc = accuracy_score(labels, preds)
    print(f"Overall Accuracy: {acc:.4f}")

# Evaluate on test set
edge_classification_report(model, x, edge_index, test_edges, test_labels, le)




📊 Edge-wise Classification Report:
              precision    recall  f1-score   support

         AdG      0.127     0.001     0.002     20360
         AeG      0.718     0.999     0.835    105481
         AuG      0.000     0.000     0.000     19549
         CbG      0.699     0.753     0.725      2342
         CdG      0.527     0.440     0.480      4085
         CpD      0.200     0.014     0.026        71
         CrC      0.849     0.940     0.892      1275
         CtD      0.627     0.185     0.286       173
         CuG      0.444     0.560     0.495      3800
         DaG      0.586     0.498     0.539      2624
         DdG      0.357     0.059     0.101      1555
         DlA      0.509     0.643     0.568       733
         DpS      0.526     0.472     0.498       665
         DrD      0.174     0.047     0.073        86
         DuG      0.312     0.388     0.346      1587
         GcG      0.507     0.349     0.413     12395
         GiG      0.553     0.644     0.595  

#RGCN


#link Prediction Binary

In [ ]:


# ===============================
# 0) Imports & device
# ===============================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from collections import defaultdict
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===============================
# 1) Graph preprocessing (G assumed to be loaded)
# ===============================
# Node mappings
node_type_map = defaultdict(list)
node_id_map = {}
node_idx_by_type = defaultdict(dict)
for nid, data in G.nodes(data=True):
    ntype = data.get("kind", "Unknown")
    idx = len(node_type_map[ntype])
    node_type_map[ntype].append(nid)
    node_id_map[nid] = (ntype, idx)
    node_idx_by_type[ntype][nid] = idx

# Edge mappings
edge_type_map = defaultdict(list)
for u, v, data in G.edges(data=True):
    rel = data.get("metaedge", data.get("relation", None))
    if not rel or u not in node_id_map or v not in node_id_map:
        continue
    u_t, u_i = node_id_map[u]
    v_t, v_i = node_id_map[v]
    edge_type_map[(u_t, rel, v_t)].append((u_i, v_i))

print(f"Node types: {list(node_type_map.keys())}")
print(f"Edge types: {len(edge_type_map)}")

# ===============================
# 2) Build homogeneous graph tensors
# ===============================
type_offsets = {}
global_id_map = {}
offset = 0
for ntype, nodes in node_type_map.items():
    type_offsets[ntype] = offset
    for i in range(len(nodes)):
        global_id_map[(ntype, i)] = offset + i
    offset += len(nodes)
num_nodes = offset

edge_type_keys = list(edge_type_map.keys())
rel2id = {etype: i for i, etype in enumerate(edge_type_keys)}
num_relations = len(edge_type_keys)

all_src, all_dst, all_rel = [], [], []
for etype, edges in edge_type_map.items():
    rel_id = rel2id[etype]
    src_type, _, dst_type = etype
    for s_local, d_local in edges:
        all_src.append(type_offsets[src_type] + s_local)
        all_dst.append(type_offsets[dst_type] + d_local)
        all_rel.append(rel_id)

edge_index = torch.tensor([all_src, all_dst], dtype=torch.long, device=device)
edge_type  = torch.tensor(all_rel, dtype=torch.long, device=device)
print(f"Graph tensor -> nodes: {num_nodes}, edges: {edge_index.size(1)}")

# ===============================
# 3) Per-relation train/val/test splits
# ===============================
edge_type_splits = {}
for etype in edge_type_keys:
    rel_id = rel2id[etype]
    mask = (edge_type == rel_id)
    edges = edge_index[:, mask]
    n_edges = edges.size(1)
    if n_edges < 50:  # skip too small
        continue
    perm = torch.randperm(n_edges)
    n_train = int(0.8 * n_edges)
    n_val = int(0.1 * n_edges)
    train_idx = perm[:n_train]
    val_idx = perm[n_train:n_train+n_val]
    test_idx = perm[n_train+n_val:]
    edge_type_splits[etype] = {
        "train": edges[:, train_idx],
        "val": edges[:, val_idx],
        "test": edges[:, test_idx],
        "all": edges
    }

# ===============================
# 4) RGCN model
# ===============================
class RGCNNet(nn.Module):
    def __init__(self, num_nodes, out_dim=64, num_relations=num_relations):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, 64)
        self.rgcn1 = RGCNConv(64, 64, num_relations, num_bases=None)
        self.rgcn2 = RGCNConv(64, out_dim, num_relations, num_bases=None)

    def forward(self, node_ids, edge_index, edge_type):
        x = self.emb(node_ids)
        x = F.relu(self.rgcn1(x, edge_index, edge_type))
        x = self.rgcn2(x, edge_index, edge_type)
        return x

def edge_scores(z, eidx):
    return torch.sigmoid((z[eidx[0]] * z[eidx[1]]).sum(dim=-1))

# ===============================
# 5) Negative sampling
# ===============================
def build_pools_and_forbidden(edge_type_splits, node_type_map, type_offsets):
    rel_resources = {}
    for rel_key, splits in edge_type_splits.items():
        src_t, _, dst_t = rel_key
        src_pool = list(range(type_offsets[src_t], type_offsets[src_t] + len(node_type_map[src_t])))
        dst_pool = list(range(type_offsets[dst_t], type_offsets[dst_t] + len(node_type_map[dst_t])))
        forbidden = set((int(s), int(d)) for s, d in zip(splits["all"][0].tolist(), splits["all"][1].tolist()))
        rel_resources[rel_key] = {"src_pool": src_pool, "dst_pool": dst_pool, "forbidden": forbidden}
    return rel_resources

def sample_type_consistent_negs(num_samples, src_pool, dst_pool, forbidden_set, device):
    src_pool_t = torch.tensor(src_pool, device=device)
    dst_pool_t = torch.tensor(dst_pool, device=device)
    neg_src, neg_dst = [], []
    while len(neg_src) < num_samples:
        s = src_pool_t[torch.randint(0, len(src_pool_t), (num_samples*2,))]
        d = dst_pool_t[torch.randint(0, len(dst_pool_t), (num_samples*2,))]
        pairs = [(int(si), int(di)) for si, di in zip(s, d)]
        filtered = [p for p in pairs if p not in forbidden_set]
        if len(filtered) > 0:
            ns = [p[0] for p in filtered][:num_samples - len(neg_src)]
            nd = [p[1] for p in filtered][:num_samples - len(neg_dst)]
            neg_src.extend(ns)
            neg_dst.extend(nd)
    return torch.tensor([neg_src, neg_dst], device=device)

rel_resources = build_pools_and_forbidden(edge_type_splits, node_type_map, type_offsets)

# ===============================
# 6) Train unified RGCN
# ===============================
node_ids = torch.arange(num_nodes, device=device)
model = RGCNNet(num_nodes, out_dim=64, num_relations=num_relations).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
EPOCHS = 20

for epoch in range(1, EPOCHS+1):
    model.train()
    optimizer.zero_grad()
    z = model(node_ids, edge_index, edge_type)
    pos_edges_all, neg_edges_all = [], []
    for rel_key, splits in edge_type_splits.items():
        pos_e = splits["train"]
        if pos_e.size(1) == 0:
            continue
        N = pos_e.size(1)
        neg_e = sample_type_consistent_negs(N,
                    rel_resources[rel_key]["src_pool"],
                    rel_resources[rel_key]["dst_pool"],
                    rel_resources[rel_key]["forbidden"],
                    device=device)
        pos_edges_all.append(pos_e)
        neg_edges_all.append(neg_e)
    pos_edges_all = torch.cat(pos_edges_all, dim=1)
    neg_edges_all = torch.cat(neg_edges_all, dim=1)
    pos_scores = edge_scores(z, pos_edges_all)
    neg_scores = edge_scores(z, neg_edges_all)
    loss = -(pos_scores+1e-15).log().mean() - (1-neg_scores+1e-15).log().mean()
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# ===============================
# 7) Evaluation per relation
# ===============================
model.eval()
results = []
with torch.no_grad():
    z = model(node_ids, edge_index, edge_type)
    for rel_key, splits in edge_type_splits.items():
        N = splits["test"].size(1)
        if N == 0: continue
        neg_e = sample_type_consistent_negs(N,
                    rel_resources[rel_key]["src_pool"],
                    rel_resources[rel_key]["dst_pool"],
                    rel_resources[rel_key]["forbidden"],
                    device=device)
        y_true = torch.cat([torch.ones(N, device=device), torch.zeros(N, device=device)])
        scores = torch.cat([edge_scores(z, splits["test"]), edge_scores(z, neg_e)])
        y = y_true.cpu().numpy()
        s = scores.cpu().numpy()
        auc = roc_auc_score(y, s)
        thr = 0.5
        preds = (s > thr).astype(int)
        p = precision_score(y, preds, zero_division=0)
        r = recall_score(y, preds, zero_division=0)
        f1 = f1_score(y, preds, zero_division=0)
        results.append((rel_key, auc, p, r, f1, thr))

print("\n📊 Final evaluation per edge type:")
for rel_key, auc, p, r, f1, thr in results:
    print(f"{rel_key}: AUC={auc:.4f}, P={p:.3f}, R={r:.3f}, F1={f1:.3f}, Thr={thr:.2f}")

# ===============================
# 8) Overall evaluation
# ===============================
all_pos, all_neg = [], []
with torch.no_grad():
    for rel_key, splits in edge_type_splits.items():
        N = splits["test"].size(1)
        if N == 0: continue
        neg_e = sample_type_consistent_negs(N,
                    rel_resources[rel_key]["src_pool"],
                    rel_resources[rel_key]["dst_pool"],
                    rel_resources[rel_key]["forbidden"],
                    device=device)
        all_pos.append(splits["test"])
        all_neg.append(neg_e)

all_pos = torch.cat(all_pos, dim=1)
all_neg = torch.cat(all_neg, dim=1)

with torch.no_grad():
    z = model(node_ids, edge_index, edge_type)
    scores_pos = edge_scores(z, all_pos)
    scores_neg = edge_scores(z, all_neg)

y_true = torch.cat([torch.ones(scores_pos.numel(), device=device),
                    torch.zeros(scores_neg.numel(), device=device)])
scores_all = torch.cat([scores_pos, scores_neg], dim=0)
thr_overall = 0.5
preds_overall = (scores_all > thr_overall).int()

auc_overall = roc_auc_score(y_true.cpu().numpy(), scores_all.cpu().numpy())
p_overall   = precision_score(y_true.cpu().numpy(), preds_overall.cpu().numpy(), zero_division=0)
r_overall   = recall_score(y_true.cpu().numpy(), preds_overall.cpu().numpy(), zero_division=0)
f1_overall  = f1_score(y_true.cpu().numpy(), preds_overall.cpu().numpy(), zero_division=0)

print("\n🧮 Overall performance across all relations:")
print(f"AUC = {auc_overall:.4f}, Precision = {p_overall:.3f}, Recall = {r_overall:.3f}, F1 = {f1_overall:.3f}")


Using device: cuda
Node types: ['Anatomy', 'Biological Process', 'Cellular Component', 'Compound', 'Disease', 'Gene', 'Molecular Function', 'Pathway', 'Pharmacologic Class', 'Side Effect', 'Symptom', 'Unknown']
Edge types: 20
Graph tensor -> nodes: 47033, edges: 1379933
Epoch 1 | Loss: 27.4898
Epoch 5 | Loss: 21.2778
Epoch 10 | Loss: 11.6912
Epoch 15 | Loss: 6.0136
Epoch 20 | Loss: 3.5757

📊 Final evaluation per edge type:
('Anatomy', 'AdG', 'Gene'): AUC=0.8298, P=0.708, R=0.886, F1=0.787, Thr=0.50
('Anatomy', 'AeG', 'Gene'): AUC=0.8186, P=0.706, R=0.834, F1=0.765, Thr=0.50
('Anatomy', 'AuG', 'Gene'): AUC=0.8340, P=0.705, R=0.895, F1=0.789, Thr=0.50
('Compound', 'CrC', 'Compound'): AUC=0.7274, P=0.541, R=0.926, F1=0.683, Thr=0.50
('Compound', 'CtD', 'Disease'): AUC=0.6477, P=0.513, R=0.789, F1=0.622, Thr=0.50
('Compound', 'CbG', 'Gene'): AUC=0.6792, P=0.603, R=0.715, F1=0.655, Thr=0.50
('Compound', 'CuG', 'Gene'): AUC=0.6748, P=0.588, R=0.711, F1=0.644, Thr=0.50
('Compound', 'CdG', 'Ge

In [4]:
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.2.0+cu118.html


Looking in links: https://data.pyg.org/whl/torch-2.2.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 42.3 MB/s eta 0:00:00


## predicting the type of link using RGCN

In [5]:
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.2.0+cu118.html



Looking in links: https://data.pyg.org/whl/torch-2.2.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 24.5 MB/s eta 0:00:00


In [6]:
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.2.0+cu118.html


Looking in links: https://data.pyg.org/whl/torch-2.2.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 19.5 MB/s eta 0:00:00


In [7]:

!pip install torch-spline-conv -f https://data.pyg.org/whl/torch-2.2.0+cu118.html
!pip install torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.2.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 902.1/902.1 kB 7.8 MB/s eta 0:00:00
  Using cached torch_geometric-2.6.1-py3-none-any.whl.metadata (63 kB)
Using cached torch_geometric-2.6.1-py3-none-any.whl (1.1 MB)


In [8]:

# -----------------------------
# 1️⃣ Imports
# -----------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import networkx as nx
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv

# -----------------------------
# 2️⃣ Load graph
# -----------------------------
G = nx.read_graphml("hetionet.graphml")
G_u = G  # RGCN needs undirected edges for message passing

# Map nodes to integer indices
node_id_map = {nid:i for i,nid in enumerate(G_u.nodes())}
num_nodes = len(node_id_map)

# Encode edge types
edge_types = [d.get("relation") for u,v,d in G_u.edges(data=True)]
le = LabelEncoder()
le.fit(edge_types)
num_rels = len(le.classes_)

# Build edge index and relation IDs
edges_src = []
edges_dst = []
rel_ids = []
for u,v,d in G_u.edges(data=True):
    edges_src.append(node_id_map[u])
    edges_dst.append(node_id_map[v])
    rel_ids.append(le.transform([d.get("relation")])[0])

edges_src = np.array(edges_src)
edges_dst = np.array(edges_dst)
rel_ids = np.array(rel_ids)

# -----------------------------
# 3️⃣ Prepare PyG data
# -----------------------------
# Simple node features: degree
x = torch.tensor([[G_u.degree(n)] for n in G_u.nodes()], dtype=torch.float)

edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
edge_type = torch.tensor(rel_ids, dtype=torch.long)

# Train / Val / Test split
idx = np.arange(edge_index.shape[1])
np.random.shuffle(idx)
n_train = int(0.8*len(idx))
n_val = int(0.1*len(idx))

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train+n_val]
test_idx = idx[n_train+n_val:]

train_edges = edge_index[:, train_idx]
train_labels = edge_type[train_idx]

val_edges = edge_index[:, val_idx]
val_labels = edge_type[val_idx]

test_edges = edge_index[:, test_idx]
test_labels = edge_type[test_idx]

data = Data(x=x, edge_index=edge_index)

# -----------------------------
# 4️⃣ Define Edge RGCN Model
# -----------------------------
class EdgeRGCN(nn.Module):
    def __init__(self, in_dim, h_dim, num_classes, num_rels, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(RGCNConv(in_dim, h_dim, num_rels))
        for _ in range(num_layers-1):
            self.convs.append(RGCNConv(h_dim, h_dim, num_rels))
        self.fc = nn.Linear(2*h_dim, num_classes)  # edge classification

    def forward(self, x, edge_index, edge_type, edge_pairs):
        # Node embeddings
        for conv in self.convs:
            x = F.relu(conv(x, edge_index, edge_type))
        src = x[edge_pairs[0]]
        dst = x[edge_pairs[1]]
        out = self.fc(torch.cat([src, dst], dim=1))
        return out

# -----------------------------
# 5️⃣ Training Setup
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = EdgeRGCN(in_dim=1, h_dim=64, num_classes=num_rels, num_rels=num_rels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

x = x.to(device)
edge_index = edge_index.to(device)
edge_type = edge_type.to(device)
train_edges = train_edges.to(device)
train_labels = train_labels.to(device)
val_edges = val_edges.to(device)
val_labels = val_labels.to(device)
test_edges = test_edges.to(device)
test_labels = test_labels.to(device)

# -----------------------------
# 6️⃣ Training Loop
# -----------------------------
EPOCHS = 20
best_val_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    out = model(x, edge_index, edge_type, train_edges)
    loss = criterion(out, train_labels)
    loss.backward()
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_out = model(x, edge_index, edge_type, val_edges)
        val_pred = val_out.argmax(dim=1)
        val_f1 = f1_score(val_labels.cpu(), val_pred.cpu(), average="macro")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict()
    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Val Macro F1: {val_f1:.4f}")

# Load best model
model.load_state_dict(best_state)

# -----------------------------
# 7️⃣ Test Evaluation
# -----------------------------
model.eval()
with torch.no_grad():
    test_out = model(x, edge_index, edge_type, test_edges)
    test_pred = test_out.argmax(dim=1)

acc = accuracy_score(test_labels.cpu(), test_pred.cpu())
precision = precision_score(test_labels.cpu(), test_pred.cpu(), average="macro", zero_division=0)
recall = recall_score(test_labels.cpu(), test_pred.cpu(), average="macro", zero_division=0)
f1 = f1_score(test_labels.cpu(), test_pred.cpu(), average="macro", zero_division=0)

print("\n📊 RGCN Test Performance:")
print(f"Accuracy = {acc:.4f}, Precision = {precision:.4f}, Recall = {recall:.4f}, F1 = {f1:.4f}")


/usr/local/lib/python3.12/dist-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/usr/local/lib/python3.12/dist-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /usr/local/lib/python3.12/dist-packages/torch_cluster/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
/usr/local/lib/python3.12/dist-packages/torch_geometric/typing.py:113: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: /usr/local/lib/python3.12/dist-packages/torch_spline_conv/_version_cuda.so: undefined symbol: _Z

Epoch 1 | Loss: 4138.0840 | Val Macro F1: 0.0604
Epoch 2 | Loss: 2257.9172 | Val Macro F1: 0.0897
Epoch 3 | Loss: 2171.0762 | Val Macro F1: 0.1292
Epoch 4 | Loss: 1007.4515 | Val Macro F1: 0.1918
Epoch 5 | Loss: 545.6582 | Val Macro F1: 0.1822
Epoch 6 | Loss: 824.1008 | Val Macro F1: 0.1929
Epoch 7 | Loss: 692.2413 | Val Macro F1: 0.2027
Epoch 8 | Loss: 464.1641 | Val Macro F1: 0.2432
Epoch 9 | Loss: 421.8494 | Val Macro F1: 0.2498
Epoch 10 | Loss: 501.8280 | Val Macro F1: 0.2646
Epoch 11 | Loss: 387.6953 | Val Macro F1: 0.3097
Epoch 12 | Loss: 277.0844 | Val Macro F1: 0.3297
Epoch 13 | Loss: 287.9215 | Val Macro F1: 0.3344
Epoch 14 | Loss: 247.8893 | Val Macro F1: 0.3491
Epoch 15 | Loss: 174.1919 | Val Macro F1: 0.3392
Epoch 16 | Loss: 166.0936 | Val Macro F1: 0.3488
Epoch 17 | Loss: 140.3228 | Val Macro F1: 0.3679
Epoch 18 | Loss: 117.4589 | Val Macro F1: 0.3911
Epoch 19 | Loss: 89.7480 | Val Macro F1: 0.3645
Epoch 20 | Loss: 97.1044 | Val Macro F1: 0.3751

📊 RGCN Test Performance:
A

In [10]:
from sklearn.metrics import classification_report
import numpy as np

# Edge-wise predictions
model.eval()
with torch.no_grad():
    test_out = model(x, edge_index, edge_type, test_edges)
    test_pred = test_out.argmax(dim=1)

# Convert to CPU numpy
test_pred = test_pred.cpu().numpy()
test_true = test_labels.cpu().numpy()

# Only include labels that actually appear in the test set
unique_labels = np.unique(np.concatenate([test_true, test_pred]))
target_names = [le.classes_[i] for i in unique_labels]

# Generate per-class report
report = classification_report(
    test_true, test_pred, labels=unique_labels, target_names=target_names, digits=3, zero_division=0
)
print("\n📊 Edge-wise Classification Report:")
print(report)



📊 Edge-wise Classification Report:
              precision    recall  f1-score   support

         AdG      0.160     0.240     0.192     10308
         AeG      0.732     0.789     0.760     52543
         AuG      0.186     0.065     0.097      9814
         CbG      0.182     0.330     0.235      1140
         CdG      0.368     0.880     0.519      2073
         CpD      0.403     0.771     0.529        35
         CrC      0.000     0.000     0.000       681
         CtD      0.795     0.461     0.583        76
         CuG      0.372     0.083     0.136      1880
         DaG      0.020     0.006     0.009      1219
         DdG      0.246     0.258     0.252       796
         DlA      0.973     0.706     0.818       357
         DpS      1.000     0.738     0.849       321
         DrD      0.029     0.136     0.048        59
         DuG      0.219     0.066     0.102       829
         GcG      0.619     0.354     0.450      6221
         GiG      0.333     0.810     0.472  